In [1]:
import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt

In [5]:
from scipy.stats import chi2_contingency
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score

# feature selection

## filter methods

In [38]:
def heatMap(data, colorbarLabel="measurement of correlation", threshold=0.5):
    names = list(data.columns) if hasattr(data, "columns") else None
    data = np.asarray(data)

    fig, ax = plt.subplots()
    im = ax.imshow(data, cmap="YlGn")
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.set_ylabel(colorbarLabel, rotation=-90, va="bottom")

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f"{data[i, j]:.2f}", ha="center", va="center",
                    color="black" if abs(data[i, j]) < threshold else "white")

    ax.set_xticks(np.arange(data.shape[1]))
    ax.set_yticks(np.arange(data.shape[0]))
    if names is not None:
        ax.set_xticklabels(names, rotation=45, ha="right")
        ax.set_yticklabels(names)

    return fig, ax

In [15]:
def chiSquareTest(input_col, output_col, df, alpha=0.05, verbose=False):
    result = {}
    table = pd.crosstab(df[output_col], df[output_col])
    result['table'] = table
    if verbose:
        print(table, "\n")
    
    chi2, p, dof, expected = chi2_contingency(table)
    result['statistics'] = {'chi2' : chi2, 'p' : p, 'dof' : dof, 'expected' : expected}
    if verbose:
        print(f"chi2={chi2:.2f}, p={p:.3g}, dof={dof}\n")
    
    if verbose:    
        print(pd.DataFrame(expected, index=table.index, columns=table.columns))
    result['min_expected'] = expected.min()
    if verbose:
        print("min expected:", expected.min(), "\n")

    if p < alpha:
        statement = f"\nReject H0: {input_col} and {output_col} are dependent"
    else:
        statement = f"\nFail to reject H0: {input_col} and {output_col} are Independent"
    result['statement'] = statement
    if verbose:
        print(statement, "\n")

    return result

In [13]:
def calculateNormalMutualInformation(input_col, output_col, df):
    data = df[[input_col, output_col]].dropna()
    x, y = data.iloc[:, 0], data.iloc[:, 1]
    return normalized_mutual_info_score(x, y)